# HotpotQA Baseline

Baseline model using DistilBERT with sentence transformers for retrieval.

In [29]:
%pip install -q -r ../requirements.txt tf-keras

python(81067) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports

In [30]:
import json
import os
import sys
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
from tqdm import tqdm

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.modules['tensorflow'] = None
sys.modules['tf_keras'] = None
sys.modules['keras'] = None

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
    USE_SENTENCE_TRANSFORMERS = True
except ImportError:
    USE_SENTENCE_TRANSFORMERS = False

sys.path.append(str(Path.cwd().parent))

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: mps


## Configuration

In [31]:
CONFIG = {
    'data_dir': Path('../data'),
    'results_dir': Path('../results'),
    'model_name': 'distilbert-base-uncased-distilled-squad',
    'max_length': 512,
    'doc_stride': 128,
    'batch_size': 8,
    'learning_rate': 3e-5,
    'num_epochs': 3,
    'top_k_retrieval': 10,
    'max_context_length': 400,
    'device': device
}

CONFIG['results_dir'].mkdir(exist_ok=True)

## Data Loading

In [32]:
def load_hotpot_data(file_path: Path) -> List[Dict]:
    with open(file_path, 'r') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} examples from {file_path.name}")
    return data

def extract_paragraphs(context: List) -> List[str]:
    paragraphs = []
    for para in context:
        if isinstance(para, list) and len(para) >= 2:
            title = para[0] if para[0] else ''
            sentences = para[1] if isinstance(para[1], list) else []
            text = ' '.join(sentences)
            paragraphs.append(f"{title} {text}")
        elif isinstance(para, dict):
            title = para.get('title', '')
            sentences = para.get('sentences', [])
            text = ' '.join(sentences)
            paragraphs.append(f"{title} {text}")
    return paragraphs

def format_context_for_qa(context: List[Dict]) -> str:
    paragraphs = extract_paragraphs(context)
    return ' '.join(paragraphs)

dev_fullwiki = load_hotpot_data(CONFIG['data_dir'] / 'hotpot_dev_fullwiki_v1.json')
dev_distractor = load_hotpot_data(CONFIG['data_dir'] / 'hotpot_dev_distractor_v1.json')

Loaded 7405 examples from hotpot_dev_fullwiki_v1.json
Loaded 7405 examples from hotpot_dev_distractor_v1.json


## Retrieval

In [ ]:
class ImprovedRetriever:
    def __init__(self, top_k: int = 10, use_sentence_transformers: bool = True):
        self.top_k = top_k
        self.paragraphs = []
        self.use_sentence_transformers = use_sentence_transformers and USE_SENTENCE_TRANSFORMERS
        
        if self.use_sentence_transformers:
            self.encoder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
            self.vectors = None
        else:
            self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
            self.vectors = None
        
    def index(self, all_paragraphs: List[str]):
        self.paragraphs = all_paragraphs
        
        if self.use_sentence_transformers:
            batch_size = 32
            self.vectors = []
            for i in tqdm(range(0, len(self.paragraphs), batch_size), desc="Indexing"):
                batch = self.paragraphs[i:i+batch_size]
                batch_vectors = self.encoder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
                self.vectors.append(batch_vectors)
            self.vectors = np.vstack(self.vectors)
        else:
            self.vectors = self.vectorizer.fit_transform(self.paragraphs)
        
    def retrieve(self, question: str, top_k: int = None) -> List[str]:
        if top_k is None:
            top_k = self.top_k
            
        if self.vectors is None or len(self.paragraphs) == 0:
            return []
        
        if self.use_sentence_transformers:
            question_vec = self.encoder.encode([question], convert_to_numpy=True)
            similarities = cosine_similarity(question_vec, self.vectors)[0]
        else:
            question_vec = self.vectorizer.transform([question])
            similarities = cosine_similarity(question_vec, self.vectors)[0]
        
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [self.paragraphs[i] for i in top_indices]

all_paragraphs_fullwiki = set()
for example in dev_fullwiki:
    paragraphs = extract_paragraphs(example['context'])
    all_paragraphs_fullwiki.update(paragraphs)

all_paragraphs_fullwiki = list(all_paragraphs_fullwiki)
print(f"Building corpus: {len(all_paragraphs_fullwiki)} unique paragraphs")

retriever = ImprovedRetriever(top_k=CONFIG['top_k_retrieval'], use_sentence_transformers=True)
retriever.index(all_paragraphs_fullwiki)

Building retrieval corpus for FullWiki...
Total unique paragraphs in FullWiki dev set: 66573
Indexing 66573 paragraphs...
✓ Indexed 66573 paragraphs with TF-IDF


Building corpus: 66573 unique paragraphs


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexing: 100%|██████████| 2081/2081 [11:52<00:00,  2.92it/s]


## Model

In [34]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG['model_name'])
model.to(CONFIG['device'])

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

## Inference

In [35]:
def is_yes_no_question(question: str) -> bool:
    question_lower = question.lower().strip()
    yes_no_starters = [
        'were', 'was', 'are', 'is', 'did', 'do', 'does', 'have', 'has', 'had',
        'can', 'could', 'will', 'would', 'should', 'may', 'might'
    ]
    return any(question_lower.startswith(starter + ' ') for starter in yes_no_starters)

def predict_answer(question: str, context: str, model, tokenizer, device, max_answer_length: int = 100) -> Tuple[str, float, float]:
    if not context or len(context.strip()) == 0:
        return "noanswer", 0.0, 0.0
    
    is_yes_no = is_yes_no_question(question)
    
    inputs = tokenizer(
        question,
        context,
        truncation=True,
        max_length=CONFIG['max_length'],
        padding='max_length',
        return_tensors='pt'
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
    
    input_ids = inputs['input_ids'][0]
    sep_token_id = tokenizer.sep_token_id
    question_end = (input_ids == sep_token_id).nonzero(as_tuple=True)[0]
    
    if len(question_end) > 0:
        min_start = question_end[0].item() + 1
    else:
        min_start = 1
    
    valid_start_logits = start_logits[0, min_start:]
    valid_end_logits = end_logits[0, min_start:]
    
    start_idx = torch.argmax(valid_start_logits).item() + min_start
    end_idx = torch.argmax(valid_end_logits).item() + min_start
    
    if end_idx < start_idx:
        end_idx = start_idx
    
    answer_tokens = input_ids[start_idx:end_idx+1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    
    if len(answer) > max_answer_length:
        sentences = answer.split('.')
        if len(sentences) > 0 and len(sentences[0]) <= max_answer_length:
            answer = sentences[0].strip()
        else:
            answer = answer[:max_answer_length].rsplit(' ', 1)[0].strip()
    
    if is_yes_no:
        answer_lower = answer.lower()
        if any(word in answer_lower for word in ['yes', 'true', 'correct', 'same', 'both']):
            answer = "yes"
        elif any(word in answer_lower for word in ['no', 'false', 'incorrect', 'different', 'not']):
            answer = "no"
    
    answer = ' '.join(answer.split()).strip()
    
    if not answer or len(answer) == 0:
        answer = "noanswer"
    
    start_score = torch.softmax(start_logits, dim=1)[0, start_idx].item()
    end_score = torch.softmax(end_logits, dim=1)[0, end_idx].item()
    
    return answer, start_score, end_score

## FullWiki Evaluation

In [36]:
predictions_fullwiki = {
    'answer': {},
    'sp': {}
}

eval_subset = dev_fullwiki[:100]

for example in tqdm(eval_subset, desc="FullWiki evaluation"):
    question = example['question']
    example_id = example['_id']
    
    retrieved_paragraphs = retriever.retrieve(question, top_k=CONFIG['top_k_retrieval'])
    context = ' '.join(retrieved_paragraphs)
    
    answer, _, _ = predict_answer(question, context, model, tokenizer, CONFIG['device'])
    
    predictions_fullwiki['answer'][example_id] = answer
    predictions_fullwiki['sp'][example_id] = []

FullWiki evaluation: 100%|██████████| 100/100 [00:08<00:00, 12.25it/s]


## FullWiki Results

In [37]:
pred_file_fullwiki = CONFIG['results_dir'] / 'baseline_fullwiki_predictions.json'
with open(pred_file_fullwiki, 'w') as f:
    json.dump(predictions_fullwiki, f, indent=2)

gold_file_fullwiki = CONFIG['results_dir'] / 'baseline_fullwiki_gold.json'
with open(gold_file_fullwiki, 'w') as f:
    json.dump(eval_subset, f, indent=2)

import subprocess
result = subprocess.run(
    ['python', '../scripts/hotpot_evaluate_v1.py', str(pred_file_fullwiki), str(gold_file_fullwiki)],
    capture_output=True,
    text=True
)
print(result.stdout)

python(81271) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


{'em': 0.09, 'f1': 0.2020543345543345, 'prec': 0.2002705627705628, 'recall': 0.2367063492063492, 'sp_em': 0.0, 'sp_f1': 0.0, 'sp_prec': 0.0, 'sp_recall': 0.0, 'joint_em': 0.0, 'joint_f1': 0.0, 'joint_prec': 0.0, 'joint_recall': 0.0}



## Distractor Evaluation

In [38]:
predictions_distractor = {
    'answer': {},
    'sp': {}
}

eval_subset_distractor = dev_distractor[:100]

for example in tqdm(eval_subset_distractor, desc="Distractor evaluation"):
    question = example['question']
    example_id = example['_id']
    
    context = format_context_for_qa(example['context'])
    answer, _, _ = predict_answer(question, context, model, tokenizer, CONFIG['device'])
    
    predictions_distractor['answer'][example_id] = answer
    predictions_distractor['sp'][example_id] = []

pred_file_distractor = CONFIG['results_dir'] / 'baseline_distractor_predictions.json'
with open(pred_file_distractor, 'w') as f:
    json.dump(predictions_distractor, f, indent=2)

gold_file_distractor = CONFIG['results_dir'] / 'baseline_distractor_gold.json'
with open(gold_file_distractor, 'w') as f:
    json.dump(eval_subset_distractor, f, indent=2)

result = subprocess.run(
    ['python', '../scripts/hotpot_evaluate_v1.py', str(pred_file_distractor), str(gold_file_distractor)],
    capture_output=True,
    text=True
)
print(result.stdout)

Distractor evaluation: 100%|██████████| 100/100 [00:03<00:00, 33.04it/s]


{'em': 0.12, 'f1': 0.1962063492063492, 'prec': 0.19305627705627704, 'recall': 0.22426190476190477, 'sp_em': 0.0, 'sp_f1': 0.0, 'sp_prec': 0.0, 'sp_recall': 0.0, 'joint_em': 0.0, 'joint_f1': 0.0, 'joint_prec': 0.0, 'joint_recall': 0.0}



python(81291) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


## 10. Summary and Next Steps

In [39]:
print(f"Results saved:")
print(f"  FullWiki: {pred_file_fullwiki}")
print(f"  Distractor: {pred_file_distractor}")

Results saved:
  FullWiki: ../results/baseline_fullwiki_predictions.json
  Distractor: ../results/baseline_distractor_predictions.json
